In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

import time
SEED = 42
np.random.seed(SEED)
print("Imports OK ✓")

## 2. Load & Inspect Dataset

In [2]:
train_df = pd.read_csv("/kaggle/input/datasets/kaushil268/disease-prediction-using-machine-learning/Training.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/kaushil268/disease-prediction-using-machine-learning/Testing.csv")

train_df = train_df.loc[:, ~train_df.columns.str.startswith('Unnamed')]
test_df  = test_df.loc[:,  ~test_df.columns.str.startswith('Unnamed')]

print(f"Train shape : {train_df.shape}  ← {train_df['prognosis'].nunique()} diseases × {train_df.shape[0]//train_df['prognosis'].nunique()} samples each")
print(f"Test  shape : {test_df.shape}   ← only {test_df.shape[0]} rows total ({test_df['prognosis'].nunique()} diseases × 1 sample)")
print()
print("⚠ The test set has just 1 sample per disease.")
print("  Any model that memorises training data will score 100% on this test set.")
print("  Cross-validation on the full dataset gives a fairer picture.")

Train shape : (4920, 133)  ← 41 diseases × 120 samples each
Test  shape : (42, 133)   ← only 42 rows total (41 diseases × 1 sample)

⚠ The test set has just 1 sample per disease.
  Any model that memorises training data will score 100% on this test set.
  Cross-validation on the full dataset gives a fairer picture.


## 3. Feature & Label Setup

In [3]:
symptom_cols = [c for c in train_df.columns if c != 'prognosis']

# Combine train + test for full CV evaluation
full_df = pd.concat([train_df, test_df], ignore_index=True)

label_encoder = LabelEncoder()
label_encoder.fit(train_df['prognosis'])

X_full  = full_df[symptom_cols].values
y_full  = label_encoder.transform(full_df['prognosis'])

X_train = train_df[symptom_cols].values
y_train = label_encoder.transform(train_df['prognosis'])
X_test  = test_df[symptom_cols].values
y_test  = label_encoder.transform(test_df['prognosis'])

id2label = dict(enumerate(label_encoder.classes_))
print(f"Total samples for CV : {len(X_full)}")
print(f"Features             : {len(symptom_cols)}")
print(f"Classes              : {len(id2label)}")

Total samples for CV : 4962
Features             : 132
Classes              : 41


## 4. Realistic Noise-Augmented Test Set

To properly separate the algorithms, we create a **harder test set** by adding realistic noise:
- **Missing symptoms**: randomly drop 1–2 symptoms a patient reports (diagnostic uncertainty)
- **Extra symptoms**: randomly add 1 irrelevant symptom (comorbidity / misreport)

This simulates real-world conditions where patients don't always report symptoms perfectly.


In [4]:
def augment_with_noise(X, noise_level=0.08, seed=42):
    """
    Simulate real-world noise:
      - Each active symptom has `noise_level` chance of being flipped off (missed symptom)
      - Each inactive symptom has `noise_level/4` chance of being flipped on (false positive)
    """
    rng = np.random.RandomState(seed)
    X_noisy = X.copy().astype(float)
    for i in range(len(X_noisy)):
        for j in range(X_noisy.shape[1]):
            if X_noisy[i, j] == 1 and rng.rand() < noise_level:
                X_noisy[i, j] = 0   # missed symptom
            elif X_noisy[i, j] == 0 and rng.rand() < noise_level / 4:
                X_noisy[i, j] = 1   # extra/false symptom
    return X_noisy

X_noisy = augment_with_noise(X_test, noise_level=0.10)
print(f"Noisy test set created: {X_noisy.shape}")
print(f"Average symptoms per sample (original) : {X_test.sum(axis=1).mean():.2f}")
print(f"Average symptoms per sample (noisy)    : {X_noisy.sum(axis=1).mean():.2f}")

Noisy test set created: (42, 132)
Average symptoms per sample (original) : 7.79
Average symptoms per sample (noisy)    : 10.05


## 5. Algorithm Comparison — 5-Fold Cross-Validation

Using **StratifiedKFold(n_splits=5)** on the full 4962-sample dataset.  
Reporting **accuracy and macro-F1** (F1 penalises models that fail on minority classes).


In [ ]:
models = {
    "Random Forest"       : RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    "Gradient Boosting"   : GradientBoostingClassifier(n_estimators=150, random_state=SEED),
    "XGBoost"             : XGBClassifier(n_estimators=150, eval_metric='mlogloss',
                                          random_state=SEED, n_jobs=-1) if XGBOOST_AVAILABLE
                            else RandomForestClassifier(n_estimators=50, random_state=1),
    "SVM (RBF)"           : SVC(kernel='rbf', probability=True, random_state=SEED),
    "Logistic Regression" : LogisticRegression(max_iter=1000, C=1.0, random_state=SEED),
    "KNN (k=5)"           : KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Naive Bayes"         : GaussianNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []

print(f"{'Model':<22}  {'CV Acc':>8}  {'±':>6}  {'CV F1':>8}  {'±':>6}  {'Time':>6}")
print("-" * 68)

for name, clf in models.items():
    t0 = time.time()
    cv_res = cross_validate(
        clf, X_full, y_full,
        cv=cv,
        scoring=['accuracy', 'f1_macro'],
        n_jobs=-1,
    )
    elapsed = time.time() - t0
    acc_mean = cv_res['test_accuracy'].mean()
    acc_std  = cv_res['test_accuracy'].std()
    f1_mean  = cv_res['test_f1_macro'].mean()
    f1_std   = cv_res['test_f1_macro'].std()
    rows.append({
        "Model"    : name,
        "CV Acc"   : acc_mean,
        "Acc ±"    : acc_std,
        "CV F1"    : f1_mean,
        "F1 ±"     : f1_std,
        "Time (s)" : round(elapsed, 1),
    })
    print(f"{name:<22}  {acc_mean:.4f}  ±{acc_std:.4f}  {f1_mean:.4f}  ±{f1_std:.4f}  {elapsed:.1f}s")

results_df = pd.DataFrame(rows).sort_values("CV Acc", ascending=False)
print("\nFinal ranking:")
print(results_df.to_string(index=False))

### CV Accuracy with Error Bars

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = results_df["Model"].tolist()
accs  = results_df["CV Acc"].values * 100
stds  = results_df["Acc ±"].values * 100
f1s   = results_df["CV F1"].values * 100
f1std = results_df["F1 ±"].values * 100

x = np.arange(len(names))

# Accuracy with error bars
colors = ['#2ecc71' if a >= 98 else '#3498db' if a >= 90 else '#e67e22' if a >= 80 else '#e74c3c'
          for a in accs]
bars = axes[0].bar(x, accs, yerr=stds, capsize=5, color=colors, width=0.6,
                   error_kw={'elinewidth': 2})
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=30, ha='right', fontsize=9)
axes[0].set_ylim(50, 105)
axes[0].set_ylabel("CV Accuracy (%)")
axes[0].set_title("5-Fold CV Accuracy ± Std Dev")
axes[0].axhline(y=100, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
for bar, v, s in zip(bars, accs, stds):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.3,
                 f"{v:.1f}%", ha='center', va='bottom', fontsize=8)

# F1 with error bars
bars2 = axes[1].bar(x, f1s, yerr=f1std, capsize=5, color=colors, width=0.6,
                    error_kw={'elinewidth': 2})
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=30, ha='right', fontsize=9)
axes[1].set_ylim(50, 105)
axes[1].set_ylabel("Macro F1 (%)")
axes[1].set_title("5-Fold CV Macro-F1 ± Std Dev")
for bar, v, s in zip(bars2, f1s, f1std):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.3,
                 f"{v:.1f}%", ha='center', va='bottom', fontsize=8)

plt.suptitle("Algorithm Comparison — Disease Prediction (5-Fold CV)", fontsize=12)
plt.tight_layout()
plt.show()
print("Green ≥ 98% | Blue ≥ 90% | Orange ≥ 80% | Red < 80%")

### Noisy Test Set Accuracy (Real-World Robustness)

In [ ]:
print("Noisy test accuracy (10% symptom noise):")
print(f"{'Model':<22}  {'Clean Test':>11}  {'Noisy Test':>11}  {'Drop':>8}")
print("-" * 58)

noise_rows = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    clean_acc = accuracy_score(y_test,  clf.predict(X_test))
    noisy_acc = accuracy_score(y_test,  clf.predict(X_noisy))
    drop      = clean_acc - noisy_acc
    noise_rows.append({"Model": name, "Clean": clean_acc, "Noisy": noisy_acc, "Drop": drop})
    print(f"{name:<22}  {clean_acc*100:>10.1f}%  {noisy_acc*100:>10.1f}%  -{drop*100:>6.1f}%")

noise_df = pd.DataFrame(noise_rows).sort_values("Noisy", ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(noise_df))
w = 0.35
ax.bar(x - w/2, noise_df['Clean'] * 100, w, label='Clean test', color='#3498db')
ax.bar(x + w/2, noise_df['Noisy'] * 100, w, label='Noisy test (10% noise)', color='#e74c3c', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(noise_df['Model'], rotation=25, ha='right', fontsize=9)
ax.set_ylim(40, 108)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Clean vs Noisy Test Set — Robustness Check")
ax.legend()
ax.axhline(y=100, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.show()

## 6. Best Model — Full Report (Random Forest)

In [ ]:
best_model = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
print(f"Test Accuracy (42 samples): {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Macro F1                  : {f1_score(y_test, y_pred, average='macro')*100:.2f}%")
print()
print("=" * 65)
print("CLASSIFICATION REPORT (42-sample test set)")
print("=" * 65)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(16, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, xticks_rotation=90, colorbar=False, cmap='Blues')
ax.set_title("Random Forest — Confusion Matrix (42-sample test set)", fontsize=12)
plt.tight_layout()
plt.show()

### Top-20 Most Important Symptoms

In [ ]:
importances = best_model.feature_importances_
top_idx = np.argsort(importances)[::-1][:20]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(range(20), importances[top_idx] * 100, color='coral')
ax.set_xticks(range(20))
ax.set_xticklabels([symptom_cols[i] for i in top_idx], rotation=45, ha='right', fontsize=9)
ax.set_ylabel("Feature Importance (%)")
ax.set_title("Top 20 Most Predictive Symptoms (Random Forest)")
plt.tight_layout()
plt.show()

## 7. Prediction Function

In [ ]:
def predict_disease(symptom_list: list, verbose=True) -> str:
    vec = np.zeros(len(symptom_cols), dtype=int)
    unknown = []
    for sym in symptom_list:
        sym_clean = sym.strip().lower().replace(' ', '_')
        if sym_clean in symptom_cols:
            vec[symptom_cols.index(sym_clean)] = 1
        else:
            unknown.append(sym)

    proba    = best_model.predict_proba([vec])[0]
    pred_id  = np.argmax(proba)
    disease  = id2label[pred_id]
    conf     = proba[pred_id]
    top3     = [(id2label[i], proba[i]) for i in np.argsort(proba)[::-1][:3]]

    if verbose:
        print(f"  Symptoms   : {symptom_list}")
        if unknown:
            print(f"  ⚠ Unknown : {unknown}")
        print(f"  Prediction : {disease}")
        print(f"  Confidence : {conf:.1%}")
        print(f"  Top-3      : " + ", ".join(f"{d} ({p:.1%})" for d, p in top3))
    return disease

## 8. 5 Example Predictions

In [ ]:
examples = [
    {"label":"Patient 1","symptoms":["itching","skin_rash","nodal_skin_eruptions","dischromic_patches"],"expected":"Fungal infection"},
    {"label":"Patient 2","symptoms":["headache","nausea","vomiting","high_fever","chills","sweating"],"expected":"Malaria"},
    {"label":"Patient 3","symptoms":["chest_pain","breathlessness","fatigue","sweating","vomiting"],"expected":"Heart attack"},
    {"label":"Patient 4","symptoms":["back_pain","weight_loss","fatigue","loss_of_appetite","abdominal_pain","night_sweats"],"expected":"Tuberculosis"},
    {"label":"Patient 5","symptoms":["joint_pain","vomiting","fatigue","high_fever","loss_of_appetite","nausea","redness_of_eyes"],"expected":"Dengue"},
]

print("=" * 65)
print("DISEASE PREDICTION — 5 EXAMPLE PATIENTS")
print("=" * 65)
for ex in examples:
    print(f"\n{ex['label']}  (expected: {ex['expected']})")
    predict_disease(ex['symptoms'])

### Prediction Confidence Chart

In [ ]:
fig, axes = plt.subplots(1, len(examples), figsize=(18, 4))
for ax, ex in zip(axes, examples):
    vec = np.zeros(len(symptom_cols), dtype=int)
    for s in ex['symptoms']:
        sc = s.strip().lower().replace(' ','_')
        if sc in symptom_cols:
            vec[symptom_cols.index(sc)] = 1
    proba   = best_model.predict_proba([vec])[0]
    top5    = np.argsort(proba)[::-1][:5]
    labels  = [id2label[i][:16] for i in top5]
    values  = [proba[i]*100 for i in top5]
    correct = id2label[top5[0]] == ex['expected']
    colors  = ['#2ecc71' if correct else '#e74c3c'] + ['#95a5a6']*4
    ax.barh(labels[::-1], values[::-1], color=colors[::-1])
    ax.set_xlim(0, 115)
    ax.set_xlabel("Confidence (%)")
    ax.set_title(ex['label'], fontsize=9)
    for i, v in enumerate(values[::-1]):
        ax.text(v+1, i, f"{v:.1f}%", va='center', fontsize=7)
plt.suptitle("Top-5 Predictions per Patient  (green = correct, red = wrong)", fontsize=11)
plt.tight_layout()
plt.show()